# 📈 Python 1D Dynamic Programming — The Master Guide
### *From Zero to Interview-Ready*

---

> **Mental Model First:**
> 1D DP is like building a staircase one step at a time, where the cost of each step
> depends only on the steps you've already built beneath it.
> You never redo work — you store the best answer for each smaller subproblem
> in an array, then look it up when you need it.
> The key question is always: "What does dp[i] represent, and how does it depend on previous dp values?"

---

## 📋 Table of Contents

| # | Section |
|---|----------|
| 1 | [What Is 1D DP? The Visual Model](#1) |
| 2 | [Creating / Setup](#2) |
| 3 | [The Core API — All Patterns](#3) |
| 4 | [Decision Map — When To Use What](#4) |
| 5 | [Pattern 1: Climbing Stairs / Fibonacci (LC 70)](#5) |
| 6 | [Pattern 2: House Robber (LC 198)](#6) |
| 7 | [Pattern 3: House Robber II / Circular (LC 213)](#7) |
| 8 | [Pattern 4: Coin Change (LC 322)](#8) |
| 9 | [Pattern 5: Min Cost Climbing Stairs (LC 746)](#9) |
| 10 | [The 1D DP Decision Map](#10) |
| 11 | [Interview Cheat Sheet](#11) |

<a id='1'></a>
## 1. What Is 1D DP? The Visual Model

```
               1D DP — THE STAIRCASE BUILDER

  Problem: Climbing stairs (LC 70). n=5 steps, take 1 or 2 at a time.

  dp[i] = number of ways to reach step i

  Index:   0   1   2   3   4   5
  dp[]:    1   1   2   3   5   8
           ↑   ↑   ↑   ↑   ↑   ↑
          base base dp[1]  dp[2]  dp[3]  dp[4]
                    +dp[0] +dp[1] +dp[2] +dp[3]

  Recurrence: dp[i] = dp[i-1] + dp[i-2]
  (reach step i from step i-1 in 1 step, or from step i-2 in 2 steps)

  THE DP FRAMEWORK:
  1. DEFINE:     What does dp[i] store? ("minimum cost to reach step i")
  2. RECURRENCE: How does dp[i] depend on smaller subproblems?
  3. BASE CASE:  What is dp[0]? dp[1]? (the cases you know without looking back)
  4. ORDER:      Fill left to right (i depends on i-1, i-2, ...)
  5. ANSWER:     Where is the final answer? (dp[n], dp[n-1], max(dp), etc.)

  SPACE OPTIMIZATION:
  If dp[i] only needs dp[i-1] and dp[i-2], you only need 2 variables — not the whole array.
```

<a id='2'></a>
## 2. Creating / Setup

In [ ]:
# BASIC DP ARRAY — initialize, fill, read answer

# Option 1: full dp array
n = 5
dp = [0] * (n + 1)     # dp[0..n], initialized to 0
dp[0] = 1              # base case: 1 way to be at step 0 (do nothing)
dp[1] = 1              # base case: 1 way to reach step 1
for i in range(2, n + 1):
    dp[i] = dp[i-1] + dp[i-2]   # Fibonacci recurrence
print("dp array:", dp)           # [1, 1, 2, 3, 5, 8]
print("answer (ways to climb 5 stairs):", dp[n])  # 8

# Option 2: space-optimized (only keep last 2 values)
prev2, prev1 = 1, 1    # dp[0], dp[1]
for _ in range(2, n + 1):
    curr  = prev1 + prev2
    prev2 = prev1
    prev1 = curr
print("space-optimized answer:", prev1)  # 8

# Option 3: memoization (top-down) template
from functools import lru_cache

@lru_cache(maxsize=None)
def fib(i):
    if i <= 1: return 1
    return fib(i-1) + fib(i-2)

print("memoized fib(5):", fib(n))  # 8
print("DP setup patterns demonstrated.")

<a id='3'></a>
## 3. The Core API — All Patterns

```
1D DP PATTERN         RECURRENCE SHAPE           TYPICAL PROBLEMS
───────────────────────────────────────────────────────────────────────
Fibonacci variant     dp[i] = dp[i-1] + dp[i-2]  Climb stairs, decode ways
"Take or skip"        dp[i] = max(dp[i-1],        House robber, max score
                               dp[i-2] + val[i])
"Min cost to reach"   dp[i] = min(dp[i-1],        Min cost stairs, jump game
                               dp[i-2]) + cost[i]
Unbounded knapsack    dp[i] = min/max over all    Coin change, rope cut
                       valid previous states
Counting ways         dp[i] += dp[i - coin]       Coin ways, decode ways

DEFINITION TEMPLATES:
  dp[i] = max/min value achievable using the first i elements
  dp[i] = number of ways to achieve target i
  dp[i] = True/False — can we achieve state i?

THINGS YOU DO NOT DO:
❌  Skip defining dp[i] clearly — confusion about meaning → wrong recurrence
❌  Forget the base case — off-by-one errors in dp[0] or dp[1] cascade
❌  Mix up 0-indexed vs 1-indexed dp arrays vs input arrays
❌  Use recursion without memoization — exponential blowup
```

In [ ]:
# Demonstrate the 3 most common 1D DP shapes

# SHAPE 1: FIBONACCI (dp[i] depends on i-1 and i-2)
def shape_fibonacci(n):
    a, b = 1, 1
    for _ in range(n - 1):
        a, b = b, a + b
    return b
print("fibonacci shape n=6:", shape_fibonacci(6))  # 13

# SHAPE 2: TAKE OR SKIP (dp[i] = max of skip vs take)
def shape_take_or_skip(nums):
    prev2, prev1 = 0, 0
    for val in nums:
        curr  = max(prev1, prev2 + val)  # skip val vs take val + skip i-1
        prev2 = prev1
        prev1 = curr
    return prev1
print("take-or-skip [2,7,9,3,1]:", shape_take_or_skip([2,7,9,3,1]))  # 12

# SHAPE 3: UNBOUNDED KNAPSACK (inner loop over all choices)
def shape_unbounded(amount, coins):
    dp = [float('inf')] * (amount + 1)
    dp[0] = 0
    for i in range(1, amount + 1):
        for coin in coins:
            if coin <= i:
                dp[i] = min(dp[i], dp[i - coin] + 1)  # use this coin, look back
    return dp[amount]
print("unbounded knapsack amount=7 coins=[1,3,4]:", shape_unbounded(7, [1,3,4]))  # 2
print("Three core 1D DP shapes demonstrated.")

<a id='4'></a>
## 4. Decision Map — When To Use What

```
SIGNAL IN THE PROBLEM                   WHAT TO DO
───────────────────────────────────────────────────────────────────────
"ways to climb n stairs"               dp[i] = dp[i-1] + dp[i-2] (Fib)
"rob houses, skip adjacent"            dp[i] = max(dp[i-1], dp[i-2]+val)
"circular array, can't rob both ends"  run house robber twice: [0..n-2] and [1..n-1]
"minimum coins for amount"             dp[i] = min(dp[i-coin]+1) for each coin
"number of ways to make change"        dp[i] += dp[i-coin]
"minimum cost to reach top"            dp[i] = min(dp[i-1], dp[i-2]) + cost[i]
"max profit with cooldown"             dp with buy/sell/cooldown states
"longest increasing subsequence"       dp[i] = max(dp[j]+1) for j < i where nums[j]<nums[i]
"can we break string into words"       dp[i] = any dp[j] where s[j:i] in dict
```

<a id='5'></a>
## 5. 🧩 Pattern 1: Climbing Stairs — LC 70

---

```
PROBLEM:
  You can climb 1 or 2 steps at a time. How many distinct ways to reach step n?

TRICK:
  dp[i] = number of ways to reach step i.
  You can arrive from step i-1 (took 1 step) or step i-2 (took 2 steps).
  Recurrence: dp[i] = dp[i-1] + dp[i-2]. This is the Fibonacci sequence.

SLOW MOTION TRACE on n=5:

  step   i=0  i=1  i=2        i=3        i=4        i=5
  dp[]    1    1   1+1=2     1+2=3      2+3=5      3+5=8
                   ↑          ↑          ↑          ↑
              dp[1]+dp[0] dp[2]+dp[1] dp[3]+dp[2] dp[4]+dp[3]

  answer = dp[5] = 8

KEY INSIGHT:
  Every Fibonacci-shaped problem follows: the current state = sum of the two previous.
  Space optimize to 2 variables since dp[i] only needs i-1 and i-2.

TIME:  O(n) — one pass
SPACE: O(1) — two rolling variables
```

In [ ]:
def climb_stairs(n):
    """
    LC 70 — Climbing Stairs
    Approach: Fibonacci DP — dp[i] = dp[i-1] + dp[i-2], space-optimized to 2 vars.
    Args:
        n (int): number of stairs to climb.
    Returns:
        int: number of distinct ways to reach the top.
    Time:  O(n) — single forward pass
    Space: O(1) — two rolling variables, no dp array
    """
    if n <= 2:
        return n                # base cases: 1 way for n=1, 2 ways for n=2
    prev2, prev1 = 1, 2        # dp[1]=1, dp[2]=2
    for _ in range(3, n + 1):
        curr  = prev1 + prev2  # ways to reach here = from step below + two below
        prev2 = prev1          # slide the window forward
        prev1 = curr
    return prev1

# Slow motion on n=5:
# i=3: curr=2+1=3, prev2=2, prev1=3
# i=4: curr=3+2=5, prev2=3, prev1=5
# i=5: curr=5+3=8, prev2=5, prev1=8
# return 8

def test_harness(fn):
    tests = [
        (1, 1),    # 1 way
        (2, 2),    # {1+1} or {2}
        (3, 3),    # {1+1+1},{1+2},{2+1}
        (4, 5),
        (5, 8),
        (10, 89),
        (45, 1836311903),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(inputs[0])
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | n={inputs[0]} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(climb_stairs)
print("climb_stairs defined.")

<a id='6'></a>
## 6. 🧩 Pattern 2: House Robber — LC 198

---

```
PROBLEM:
  Rob houses along a street. Adjacent houses have a connected alarm — you can't
  rob two adjacent houses. Maximize total money.

TRICK:
  dp[i] = max money robbing from house 0 to i.
  For each house, you choose:
    - SKIP this house: dp[i] = dp[i-1]  (best from previous house)
    - ROB this house:  dp[i] = dp[i-2] + nums[i]  (skip adjacent, add current)
  Recurrence: dp[i] = max(dp[i-1], dp[i-2] + nums[i])

SLOW MOTION TRACE on nums=[2,7,9,3,1]:

  i=0: dp[0]=2  (only option: rob house 0)
  i=1: dp[1]=max(2, 0+7)=7  (rob 7 is better than 2)
  i=2: dp[2]=max(7, 2+9)=11 (rob house 2+house 0 = 11)
  i=3: dp[3]=max(11, 7+3)=11 (skip house 3 — 11 > 10)
  i=4: dp[4]=max(11, 11+1)=12 (rob house 4 + skip house 3)
  answer = 12 (rob houses 0,2,4 = 2+9+1=12)

KEY INSIGHT:
  "Can't take adjacent" → dp[i] depends on dp[i-2], not dp[i-1].
  Space optimize: only need 2 rolling variables.

TIME:  O(n)
SPACE: O(1)
```

In [ ]:
def rob(nums):
    """
    LC 198 — House Robber
    Approach: dp[i] = max(skip house i: dp[i-1], rob house i: dp[i-2] + nums[i]).
    Args:
        nums (List[int]): money in each house.
    Returns:
        int: maximum money robbed without triggering the alarm.
    Time:  O(n) — single forward pass
    Space: O(1) — two rolling variables
    """
    if not nums:
        return 0
    prev2, prev1 = 0, 0      # dp[i-2], dp[i-1] — nothing robbed yet
    for val in nums:
        curr  = max(prev1, prev2 + val)  # skip this house OR rob it
        prev2 = prev1                     # slide window
        prev1 = curr
    return prev1

# Slow motion on [2,7,9,3,1]:
# val=2: curr=max(0,0+2)=2,   prev2=0, prev1=2
# val=7: curr=max(2,0+7)=7,   prev2=2, prev1=7
# val=9: curr=max(7,2+9)=11,  prev2=7, prev1=11
# val=3: curr=max(11,7+3)=11, prev2=11,prev1=11
# val=1: curr=max(11,11+1)=12,prev2=11,prev1=12
# return 12

def test_harness(fn):
    tests = [
        ([1,2,3,1], 4),         # rob 0 and 2: 1+3=4
        ([2,7,9,3,1], 12),      # rob 0,2,4: 2+9+1=12
        ([1], 1),
        ([2,1], 2),
        ([2,1,1,2], 4),
        ([100,1,1,100], 200),   # rob first and last
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(inputs[0])
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | nums={inputs[0]} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(rob)
print("rob defined.")

<a id='7'></a>
## 7. 🧩 Pattern 3: House Robber II / Circular — LC 213

---

```
PROBLEM:
  Same as House Robber, but houses are arranged in a circle.
  The first and last house are adjacent — can't rob both.

TRICK:
  A circle means: you can't rob house 0 AND house n-1.
  Split into two linear subproblems and take the max:
    Case 1: rob houses [0 .. n-2]  (exclude last)
    Case 2: rob houses [1 .. n-1]  (exclude first)
  Apply standard House Robber to each range. Answer = max of both.

SLOW MOTION TRACE on nums=[2,3,2]:

  Case 1 (houses [0..1] = [2,3]): rob [2,3] → max=3
  Case 2 (houses [1..2] = [3,2]): rob [3,2] → max=3
  answer = max(3, 3) = 3

SLOW MOTION TRACE on nums=[1,2,3,1]:

  Case 1 (houses [0..2] = [1,2,3]): rob → max=4 (rob 1+3)
  Case 2 (houses [1..3] = [2,3,1]): rob → max=3 (rob 2+1)
  answer = max(4, 3) = 4

KEY INSIGHT:
  Circle constraint → two separate linear runs, take the best.
  Reuse the exact same rob() function from LC 198 on each subarray.

TIME:  O(n)
SPACE: O(1)
```

In [ ]:
def rob_circular(nums):
    """
    LC 213 — House Robber II
    Approach: Split circular array into two linear cases; reuse linear rob() on each.
    Args:
        nums (List[int]): money in each house arranged in a circle.
    Returns:
        int: maximum money robbed.
    Time:  O(n) — two linear passes over n elements
    Space: O(1) — two variables per pass
    """
    if len(nums) == 1:
        return nums[0]    # only one house — just take it

    def rob_linear(start, end):
        prev2, prev1 = 0, 0
        for i in range(start, end):
            curr  = max(prev1, prev2 + nums[i])  # skip or rob
            prev2 = prev1
            prev1 = curr
        return prev1

    # case 1: include house 0, exclude house n-1
    # case 2: exclude house 0, include house n-1
    return max(
        rob_linear(0, len(nums) - 1),   # [0 .. n-2]
        rob_linear(1, len(nums))         # [1 .. n-1]
    )

# Slow motion on [2,3,2]:
# rob_linear(0,2) on [2,3]: prev2=0,prev1=0
#   i=0: curr=max(0,0+2)=2,  p2=0,p1=2
#   i=1: curr=max(2,0+3)=3,  p2=2,p1=3  → 3
# rob_linear(1,3) on [3,2]: same logic → 3
# max(3,3) = 3

def test_harness(fn):
    tests = [
        ([2,3,2], 3),
        ([1,2,3,1], 4),
        ([1,2,3], 3),
        ([1], 1),
        ([0,0], 0),
        ([200,3,140,20,10], 340),  # rob house 0 + house 2
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(inputs[0])
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | nums={inputs[0]} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(rob_circular)
print("rob_circular defined.")

<a id='8'></a>
## 8. 🧩 Pattern 4: Coin Change — LC 322

---

```
PROBLEM:
  Given coin denominations and an amount, find the fewest coins needed.
  Return -1 if it's not possible. Coins can be reused (unbounded supply).

TRICK:
  dp[i] = minimum coins to make amount i.
  For each amount i, try every coin: if i >= coin, dp[i] = min(dp[i], dp[i-coin]+1).
  This is the UNBOUNDED KNAPSACK shape — each coin can be reused.
  Initialize dp[0]=0 (no coins to make 0), all others = infinity.

SLOW MOTION TRACE on coins=[1,5,6,9], amount=11:

  dp[0]=0, dp[1..11]=inf

  i=1:  coin=1: dp[1]=min(inf, dp[0]+1)=1
  i=5:  coin=1: dp[4]+1; coin=5: dp[0]+1=1  → dp[5]=1
  i=6:  coin=1: dp[5]+1=2; coin=5: dp[1]+1=2; coin=6: dp[0]+1=1 → dp[6]=1
  i=9:  coin=9: dp[0]+1=1 → dp[9]=1
  i=10: coin=1: dp[9]+1=2; coin=5: dp[5]+1=2; coin=9: dp[1]+1=2 → dp[10]=2
  i=11: coin=1: dp[10]+1=3; coin=5: dp[6]+1=2; coin=6: dp[5]+1=2 → dp[11]=2

  answer = dp[11] = 2  (coins: 5+6 or 6+5)

KEY INSIGHT:
  Bottom-up unbounded knapsack: for every amount, try every coin as the last used.
  dp[i] = inf means unreachable — return -1 if dp[amount] stays inf.

TIME:  O(amount * len(coins))
SPACE: O(amount)
```

In [ ]:
def coin_change(coins, amount):
    """
    LC 322 — Coin Change
    Approach: Unbounded knapsack DP — dp[i] = min coins to make amount i.
    Args:
        coins (List[int]): available coin denominations.
        amount (int): target amount.
    Returns:
        int: fewest coins needed, or -1 if impossible.
    Time:  O(amount * len(coins)) — fill each amount using every coin
    Space: O(amount) — dp array of size amount+1
    """
    dp = [float('inf')] * (amount + 1)
    dp[0] = 0               # 0 coins to make amount 0

    for i in range(1, amount + 1):
        for coin in coins:
            if coin <= i:   # can use this coin for amount i
                dp[i] = min(dp[i], dp[i - coin] + 1)  # use coin, look back

    return dp[amount] if dp[amount] != float('inf') else -1

# Slow motion on coins=[1,2,5], amount=11:
# dp = [0,inf,inf,...,inf]
# i=1: coin=1: dp[0]+1=1 → dp[1]=1
# i=2: coin=1: dp[1]+1=2; coin=2: dp[0]+1=1 → dp[2]=1
# i=5: coin=5: dp[0]+1=1 → dp[5]=1
# i=6: coin=1: dp[5]+1=2; coin=2: dp[4]+1=3; coin=5: dp[1]+1=2 → dp[6]=2
# i=11:coin=1: dp[10]+1=3; coin=2: dp[9]+1=3; coin=5: dp[6]+1=3 → dp[11]=3

def test_harness(fn):
    tests = [
        ([1,2,5], 11, 3),       # 5+5+1
        ([2], 3, -1),           # impossible
        ([1], 0, 0),            # 0 coins
        ([1], 1, 1),
        ([1,5,6,9], 11, 2),     # 5+6
        ([2,5,10,1], 27, 4),    # 10+10+5+2
    ]
    passed = 0
    for *inputs, expected in tests:
        coins, amount = inputs
        got = fn(coins, amount)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | coins={coins} amount={amount} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(coin_change)
print("coin_change defined.")

<a id='9'></a>
## 9. 🧩 Pattern 5: Min Cost Climbing Stairs — LC 746

---

```
PROBLEM:
  Each step has a cost. You can start at step 0 or 1. From any step,
  you can climb 1 or 2 steps. Pay cost[i] when you leave step i.
  Find minimum cost to reach the top (one step beyond the last).

TRICK:
  dp[i] = minimum cost to reach step i.
  You can arrive at step i from step i-1 (paid cost[i-1]) or from step i-2 (paid cost[i-2]).
  Recurrence: dp[i] = min(dp[i-1] + cost[i-1], dp[i-2] + cost[i-2])
  Answer: dp[n] — the step just beyond the array.

SLOW MOTION TRACE on cost=[10,15,20]:

  dp[0]=0, dp[1]=0  (free to start at 0 or 1)
  dp[2] = min(dp[1]+cost[1], dp[0]+cost[0]) = min(0+15, 0+10) = 10
  dp[3] = min(dp[2]+cost[2], dp[1]+cost[1]) = min(10+20, 0+15) = 15

  answer = dp[3] = 15 (start at step 1, pay 15, jump to step 3 which is the top)

SLOW MOTION TRACE on cost=[1,100,1,1,1,100,1,1,100,1]:

  Answer = 6 (path: 0→2→4→6→8→9→top, paying 1+1+1+1+1+1=6)

KEY INSIGHT:
  Cost is paid when LEAVING a step, not when arriving.
  dp extends one past the array (top = step n).

TIME:  O(n)
SPACE: O(1)
```

In [ ]:
def min_cost_climbing_stairs(cost):
    """
    LC 746 — Min Cost Climbing Stairs
    Approach: dp[i] = min cost to reach step i; extend one past array for the top.
    Args:
        cost (List[int]): cost to leave each step.
    Returns:
        int: minimum cost to reach the floor just above the last step.
    Time:  O(n) — single forward pass
    Space: O(1) — two rolling variables
    """
    n = len(cost)
    prev2, prev1 = 0, 0    # dp[0]=0, dp[1]=0 (free to start at step 0 or 1)

    for i in range(2, n + 1):
        # arrive at step i from step i-1 (pay cost[i-1]) or from step i-2 (pay cost[i-2])
        curr  = min(prev1 + cost[i-1], prev2 + cost[i-2])
        prev2 = prev1
        prev1 = curr

    return prev1   # dp[n] = minimum cost to reach the top

# Slow motion on cost=[10,15,20]:
# i=2: curr=min(0+15, 0+10)=10, prev2=0, prev1=10
# i=3: curr=min(10+20, 0+15)=15, prev2=10, prev1=15
# return 15

def test_harness(fn):
    tests = [
        ([10,15,20], 15),
        ([1,100,1,1,1,100,1,1,100,1], 6),
        ([0,0,0,0], 0),
        ([1,2], 1),       # start at 0, pay 1, reach top
        ([1,2,3], 2),     # start at 1, pay 2, reach top
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(inputs[0])
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | cost={inputs[0]} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(min_cost_climbing_stairs)
print("min_cost_climbing_stairs defined.")

<a id='10'></a>
## 10. The 1D DP Decision Map

```
QUESTION TYPE                         RECURRENCE                      LC PROBLEMS
──────────────────────────────────────────────────────────────────────────────────
Ways to reach step n (1 or 2 steps)   dp[i]=dp[i-1]+dp[i-2]            70
Max value skipping adjacent           dp[i]=max(dp[i-1], dp[i-2]+v)    198
Circular array variant                run linear rob twice              213
Min coins for amount (reuse coins)    dp[i]=min(dp[i-coin]+1)           322
Min cost to climb stairs              dp[i]=min(dp[i-1]+c,dp[i-2]+c)   746
Longest increasing subsequence        dp[i]=max(dp[j]+1) j<i,v[j]<v[i] 300
Word break (can split by dict)        dp[i]=any(dp[j] and s[j:i] in d)  139
Number of ways to decode string       dp[i]+=dp[i-1] or dp[i-2]        91
Jump game (can reach end?)            dp[i]=True if any dp[j] reachable 55
```

<a id='11'></a>
## 11. Interview Cheat Sheet

**1. When to reach for 1D DP:**

| Signal | What to Do |
|--------|------------|
| "ways to reach / count paths" | dp[i] = sum of previous ways |
| "max value, can't use adjacent" | dp[i] = max(skip, take + dp[i-2]) |
| "min coins / min steps" | dp[i] = min over all valid prev states |
| "circular array" | run linear DP twice, take max |
| "reuse items" | unbounded knapsack — inner loop over items |

**2. The 5-step framework — memorize this:**

```python
# STEP 1: Define dp[i] clearly
# STEP 2: Write the recurrence
# STEP 3: Set base cases (dp[0], dp[1])
# STEP 4: Fill left to right
# STEP 5: Return dp[n] (or max/min over dp)
```

**3. Common templates:**

```python
# FIBONACCI SHAPE
a, b = 1, 1
for _ in range(n - 1): a, b = b, a + b
return b

# HOUSE ROBBER SHAPE
prev2 = prev1 = 0
for val in nums:
    curr = max(prev1, prev2 + val)
    prev2, prev1 = prev1, curr
return prev1

# COIN CHANGE (UNBOUNDED KNAPSACK)
dp = [inf] * (amount + 1); dp[0] = 0
for i in range(1, amount + 1):
    for coin in coins:
        if coin <= i: dp[i] = min(dp[i], dp[i-coin] + 1)
return dp[amount] if dp[amount] != inf else -1

# CIRCULAR ARRAY (House Robber II)
def rob_range(start, end):
    p2 = p1 = 0
    for i in range(start, end):
        p2, p1 = p1, max(p1, p2 + nums[i])
    return p1
return max(rob_range(0, n-1), rob_range(1, n))
```

**4. Gotchas to not forget:**

```
❌  Confusing dp array index with input array index — be explicit about what dp[i] means
❌  Forgetting dp[0]=0 for count-ways / min-coins problems (seed the table)
❌  Not handling the single-element edge case in House Robber
❌  Circular: forgetting to run two separate passes (not one)
✅  Always write out dp[0], dp[1] before writing the loop
✅  If dp[i] only needs i-1 and i-2, use 2 variables — O(1) space
✅  Coin Change seed: dp[0]=0, dp[1..amount]=infinity (not zero!)
✅  Check dp[amount]==inf → return -1 for impossible coin change
```

## Summary Map

```
                    📈 1D DYNAMIC PROGRAMMING
                              │
           ┌──────────────────┼──────────────────┐
           │                  │                  │
       FIBONACCI          TAKE/SKIP          UNBOUNDED
       SHAPE              SHAPE              KNAPSACK
       dp[i]=             dp[i]=max(         dp[i]=
       dp[i-1]+           dp[i-1],           min over
       dp[i-2]            dp[i-2]+val)       all coins
       LC 70              LC 198             LC 322
                              │
                         CIRCULAR
                         VARIANT
                         run twice
                         LC 213
           │
       MIN COST
       VARIANT
       dp[i]=min(
       prev+cost)
       LC 746

CORE DP FRAMEWORK:
  1. Define dp[i]  2. Write recurrence  3. Base cases
  4. Fill left→right  5. Return dp[n] or max/min
  Optimize: if dp[i] depends on ≤2 prev → 2 variables, O(1) space
```

---
*End of 1D Dynamic Programming Master Guide — Sean Edition*